# Session 2 · LLM main runs (3 seeds per augmentation level)

Settings: **Accelerator GPU T4 x2**, **Internet On**. Inputs: the `raid-processed` dataset
**and the output of the Session 1 notebook** (Add Input → Your Work → Notebooks).
Run with **Save Version → Save & Run All (Commit)**. About 5.5 hours per run of this notebook.

Run it **twice**, because all 6 runs would take too close to Kaggle's 12-hour limit:
* **2a:** `AUG = "0"` (no augmentation)
* **2b:** `AUG = "0.5"`, with **this notebook's 2a output added as an extra input**,
  so the 2a results are carried over and everything ends up in one results file.

In [1]:
AUG = "0.5"   # 2a: "0"    2b: "0.5"

In [2]:
# Copy scripts and data from the raid-processed dataset into a working project folder
import glob, os, shutil
# (the folder that has both the data and the scripts, i.e. the dataset itself -
#  not a copy inside an earlier notebook's output)
roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train_clean.parquet", recursive=True)]
roots = [r for r in roots if glob.glob(f"{r}/**/train_llm.py", recursive=True)]
assert roots, "raid-processed dataset (with train_llm.py) not found - check the Input panel"
data_root = roots[0]
os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/data/processed", exist_ok=True)
for f in glob.glob(f"{data_root}/**/*.py", recursive=True):
    shutil.copy(f, "/kaggle/working/src/")
for f in glob.glob(f"{data_root}/**/*.parquet", recursive=True):
    shutil.copy(f, "/kaggle/working/data/processed/")
os.chdir("/kaggle/working")
print("src:", sorted(os.listdir("src")))
print("data:", sorted(os.listdir("data/processed")))
for need in ["train_llm.py", "zero_shot_binoculars.py", "metrics.py", "train_roberta.py"]:
    assert need in os.listdir("src"), f"{need} missing - upload it to the dataset (New Version)"
!pip uninstall -y -q torchao   # Kaggle's old torchao breaks recent peft; not needed here
!pip install -q peft
!nvidia-smi --query-gpu=name,memory.total --format=csv

src: ['inspect_raid.py', 'make_splits.py', 'metrics.py', 'train_classical.py', 'train_llm.py', 'train_roberta.py', 'zero_shot_binoculars.py']
data: ['test_heldout_attacks.parquet', 'test_reppen.parquet', 'test_seen.parquet', 'test_seen_attacks.parquet', 'test_unseen.parquet', 'train_attacked.parquet', 'train_clean.parquet', 'val.parquet']
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [3]:
# Learning rate chosen in session 1 (validation AUROC), plus any finished runs to resume
import glob, os, shutil
import pandas as pd
LR_MANUAL = 1e-4   # only used if the session 1 tuning file cannot be found

tuning = glob.glob("/kaggle/input/**/results/tuning/llm_efficiency.csv", recursive=True)
if tuning:
    t = pd.read_csv(tuning[0])
    BEST_LR = float(t.loc[t["val_auroc"].idxmax(), "lr"])
    os.makedirs("results/tuning", exist_ok=True)
    shutil.copy(tuning[0], "results/tuning/llm_efficiency.csv")
else:
    BEST_LR = LR_MANUAL
print("Learning rate:", BEST_LR)

os.makedirs("results/predictions", exist_ok=True)
main = [f for f in glob.glob("/kaggle/input/**/results/llm_*.csv", recursive=True)
        if "/smoke" not in f and "/tuning/" not in f]
for f in main:
    shutil.copy(f, "results/")
    print("carrying over", f)
for f in glob.glob("/kaggle/input/**/results/predictions/Qwen*.parquet", recursive=True):
    shutil.copy(f, "results/predictions/")

Learning rate: 0.0001
carrying over /kaggle/input/notebooks/souravamahajan/machine-learning/results/llm_efficiency.csv
carrying over /kaggle/input/notebooks/souravamahajan/machine-learning/results/llm_results.csv
carrying over /kaggle/input/notebooks/souravamahajan/machine-learning/results/llm_summary.csv


In [4]:
!python src/train_llm.py --lrs {BEST_LR} --aug {AUG}

Device: cuda
Tokenizing...
config.json: 100%|█████████████████████████████| 681/681 [00:00<00:00, 3.38MB/s]
tokenizer_config.json: 100%|███████████████| 7.23k/7.23k [00:00<00:00, 29.9MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 47.3MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 124MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 162MB/s]
  93,604 texts tokenized
Run aug=0.5 lr=0.0001 seed=42

model.safetensors: downloading bytes: ██                    | 91.7MB, 6.94MB/s  
model.safetensors: reconstructing file:  10%|▌     |  102MB /  988MB            
model.safetensors: downloading bytes: ██▌                   |  114MB, 9.92MB/s  
model.safetensors: downloading bytes: ██▉                   |  133MB, 11.5MB/s  
model.safetensors: downloading bytes: ███▎                  |  151MB, 12.1MB/s  
model.safetensors: downloading bytes: ██████▉               |  314MB, 22.8MB/s  
model.safetensors: downloa